# 02 · Silver de población
**Responsable:** P1 · **Código:** `src/silver_poblacion.py` · **Salidas:** `silver/poblacion.parquet`, `silver/poblacion_area.parquet`, figura de la pirámide

**Qué hace**
- `poblacion`: lee el CSV de bronze como texto, normaliza el código DIVIPOLA a 5 dígitos, convierte tipos y aplica reglas: código válido, año 2020–2025, conteos no nulos ni negativos, 18–44 ≤ 18–64 ≤ total, hombres + mujeres = total y clave única municipio-año.
- `poblacion_area`: el archivo de "centros poblados" son proyecciones por **área** (cabecera, rural y total). Se verifica que cabecera + rural = total y se conservan solo cabecera y rural, en formato largo (sexo × edad). Así nunca se suma dos veces la población.
- Concilia el total de esa fuente con la población DANE oficial y deja la diferencia en `outputs/quality/conciliacion_poblacion_fuentes.csv`.

**Decisión clave:** la población municipal oficial es `poblacion_dane`; el archivo por áreas se usa para la pirámide, el % rural y la sensibilidad.

**Qué revisar al ejecutar:** 738 filas en `poblacion` (123 × 6 años), 0 rechazos esperados, y el tamaño de las diferencias en la conciliación.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [ ]:
from src.silver_poblacion import construir_poblacion, construir_poblacion_area
pob = construir_poblacion()
pob.head()

In [ ]:
areas = construir_poblacion_area(poblacion=pob)
areas.head()

In [ ]:
pl.read_csv(QUALITY / "conciliacion_poblacion_fuentes.csv").select(
    pl.col("diferencia").min().alias("min"), pl.col("diferencia").median().alias("mediana"),
    pl.col("diferencia").max().alias("max"))

In [ ]:
from src import graficos
graficos.piramide_poblacional(areas.to_pandas(), ANIO_PIRAMIDE)

In [ ]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente").is_in(["poblacion", "centros_poblados", "poblacion_area"]))